In [2]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import os

# 1. Chargement et nettoyage
def safe_load_wkt(text):
    try:
        if pd.isna(text) or text == "": return None
        geom = wkt.loads(text)
        if geom.is_empty: return None
        if geom.geom_type == 'LineString' and len(geom.coords) < 2: return None
        return geom
    except: return None

def dwell_to_seconds(dwell_str):
    try:
        parts = str(dwell_str).split(':')
        return int(parts[0])*3600 + int(parts[1])*60 + int(parts[2]) if len(parts)==3 else 0
    except: return 0

# 2. Traitement des ARRÊTS (Points pour Cartes 1, 2 et 3)
stops = pd.read_csv('Stops_0_20251128.csv')
stops['Dwell_Sec'] = stops['Dwell Time'].apply(dwell_to_seconds)
stops['Tot_Act'] = stops['Pax On'] + stops['Pax Off']

gdf_stops = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops['Stop Lon'], stops['Stop Lat']), crs="EPSG:4326"
)

# Renommage pour QGIS (limite de 10 caractères pour les Shapefiles)
gdf_stops = gdf_stops.rename(columns={
    'Trip ID': 'Trip_ID', 'Pax On': 'Pax_On', 'Pax Off': 'Pax_Off',
    'Tot_Act': 'Tot_Act', 'Occupancy': 'Occupancy', 'Dwell_Sec': 'Dwell_Sec'
})

# Sélection des colonnes utiles pour vos 4 cartes
gdf_stops = gdf_stops[['Trip_ID', 'Pax_On', 'Pax_Off', 'Tot_Act', 'Occupancy', 'Dwell_Sec', 'geometry']]
gdf_stops.to_file('stops_complete.shp')

# 3. Traitement des TRAJETS (Lignes pour Carte 4)
trips = pd.read_csv('Trips_0_20251128.csv')
trips['geometry'] = trips['Geometry (WKT)'].apply(safe_load_wkt)
gdf_trips = gpd.GeoDataFrame(trips.dropna(subset=['geometry']), geometry='geometry', crs="EPSG:4326")

gdf_trips = gdf_trips.rename(columns={
    'Trip ID': 'Trip_ID', 'Total Passengers': 'Tot_Pax', 
    'Distance': 'Dist_km', 'Revenue': 'Revenue', 'Mode': 'Mode'
})

gdf_trips = gdf_trips[['Trip_ID', 'Tot_Pax', 'Dist_km', 'Revenue', 'Mode', 'geometry']]
gdf_trips.to_file('trips_complete.shp')

print("Fichiers Shapefile générés : stops_complete.shp et trips_complete.shp")

Fichiers Shapefile générés : stops_complete.shp et trips_complete.shp
